# Advanced Problems with Solutions: Simulating `switch` Behavior in Python

This notebook contains advanced practice problems on implementing switch-like behavior in Python using dictionaries, functions, decorators, registries, defaults, dynamic registration, and dispatch patterns.

Each problem includes a complete solution and runnable examples.

## Problem 1 — Function Registry with Default Behavior

Build a function called `make_command_router()` that returns a callable router.

The router should:

- Dispatch based on a string command.
- Support commands: `"start"`, `"stop"`, and `"restart"`.
- Return strings instead of printing.
- Return `"Unknown command"` for unsupported commands.
- Avoid using `if/elif/else` inside the router itself.

Example:

```python
router = make_command_router()
router("start")    # "System starting"
router("pause")    # "Unknown command"
```

In [1]:
def make_command_router():
    commands = {
        "start": lambda: "System starting",
        "stop": lambda: "System stopping",
        "restart": lambda: "System restarting"
    }

    default = lambda: "Unknown command"

    def router(command):
        return commands.get(command, default)()

    return router


router = make_command_router()

assert router("start") == "System starting"
assert router("stop") == "System stopping"
assert router("restart") == "System restarting"
assert router("pause") == "Unknown command"

print("Problem 1 passed")

Problem 1 passed


### Explanation

A dictionary maps command names to functions. The router selects the correct function with `.get()`. If the command is missing, it uses the default function.

This is a clean switch-like pattern when each case can be represented as a simple callable.

## Problem 2 — Dynamic Registration Decorator

Create a decorator called `switcher` that allows functions to be registered by case value.

Requirements:

- The decorated base function should act as the default case.
- The decorated function should gain a `.register(case)` method.
- Calling the decorated function with a case should dispatch to the registered function.
- Registered case functions should take no arguments.

Example:

```python
@switcher
def status():
    return "unknown"

@status.register(200)
def ok():
    return "OK"

status(200)  # "OK"
status(500)  # "unknown"
```

In [2]:
def switcher(default_fn):
    registry = {"default": default_fn}

    def register(case):
        def inner(fn):
            registry[case] = fn
            return fn
        return inner

    def dispatch(case):
        fn = registry.get(case, registry["default"])
        return fn()

    dispatch.register = register
    dispatch.registry = registry
    return dispatch


@switcher
def status():
    return "unknown"


@status.register(200)
def ok():
    return "OK"


@status.register(404)
def not_found():
    return "Not Found"


assert status(200) == "OK"
assert status(404) == "Not Found"
assert status(500) == "unknown"

print("Problem 2 passed")

Problem 2 passed


### Explanation

The decorator closes over a private `registry` dictionary. Each call to `.register(case)` stores a function for that case. The returned `dispatch` function performs the lookup at runtime.

## Problem 3 — Switch with Arguments Passed to Case Functions

Improve the previous `switcher` so that registered functions can receive arbitrary positional and keyword arguments.

Requirements:

- The first argument is the case.
- Remaining arguments are passed to the selected function.
- The default function should receive the same remaining arguments.

Example:

```python
@argument_switcher
def operation(x, y):
    return None

@operation.register("add")
def add(x, y):
    return x + y

operation("add", 10, 5)  # 15
```

In [3]:
def argument_switcher(default_fn):
    registry = {"default": default_fn}

    def register(case):
        def inner(fn):
            registry[case] = fn
            return fn
        return inner

    def dispatch(case, *args, **kwargs):
        fn = registry.get(case, registry["default"])
        return fn(*args, **kwargs)

    dispatch.register = register
    dispatch.registry = registry
    return dispatch


@argument_switcher
def operation(x, y):
    return f"Unsupported operation for values {x} and {y}"


@operation.register("add")
def add(x, y):
    return x + y


@operation.register("multiply")
def multiply(x, y):
    return x * y


assert operation("add", 10, 5) == 15
assert operation("multiply", 10, 5) == 50
assert operation("divide", 10, 5) == "Unsupported operation for values 10 and 5"

print("Problem 3 passed")

Problem 3 passed


### Explanation

This pattern is more practical than a zero-argument switch because the selected behavior can operate on input data. The dispatch key is separated from the function arguments.

## Problem 4 — Multi-Case Registration

Extend the switch decorator so that a single function can be registered for multiple case values at once.

Requirements:

- `.register(*cases)` should accept one or more cases.
- The same function should be stored for each case.
- Calling the router with any registered case should call the shared function.

Example:

```python
@multi_switcher
def classify(value):
    return "other"

@classify.register("y", "yes", "true", "1")
def truthy(value):
    return "truthy"
```

In [4]:
def multi_switcher(default_fn):
    registry = {"default": default_fn}

    def register(*cases):
        if not cases:
            raise ValueError("At least one case must be provided")

        def inner(fn):
            for case in cases:
                registry[case] = fn
            return fn
        return inner

    def dispatch(case, *args, **kwargs):
        fn = registry.get(case, registry["default"])
        return fn(case, *args, **kwargs)

    dispatch.register = register
    dispatch.registry = registry
    return dispatch


@multi_switcher
def classify(value):
    return "other"


@classify.register("y", "yes", "true", "1")
def truthy(value):
    return "truthy"


@classify.register("n", "no", "false", "0")
def falsy(value):
    return "falsy"


assert classify("yes") == "truthy"
assert classify("1") == "truthy"
assert classify("false") == "falsy"
assert classify("maybe") == "other"

print("Problem 4 passed")

Problem 4 passed


### Explanation

Many real switch statements intentionally group several labels into one branch. Python dictionaries do not support duplicate keys, but a decorator can register the same function under many keys.

## Problem 5 — Case Normalization

Create a switch decorator that supports a normalization function.

Requirements:

- The decorator factory should be called as `@normalized_switcher(normalize=str.lower)`.
- Cases should be normalized both when registering and when dispatching.
- This should allow case-insensitive command routing.

Example:

```python
@normalized_switcher(normalize=str.lower)
def command(value):
    return "unknown"

@command.register("START")
def start(value):
    return "starting"

command("start")  # "starting"
command("START")  # "starting"
```

In [5]:
def normalized_switcher(normalize=lambda x: x):
    def decorator(default_fn):
        registry = {"default": default_fn}

        def register(case):
            normalized_case = normalize(case)

            def inner(fn):
                registry[normalized_case] = fn
                return fn
            return inner

        def dispatch(case, *args, **kwargs):
            normalized_case = normalize(case)
            fn = registry.get(normalized_case, registry["default"])
            return fn(case, *args, **kwargs)

        dispatch.register = register
        dispatch.registry = registry
        return dispatch

    return decorator


@normalized_switcher(normalize=str.lower)
def command(value):
    return "unknown"


@command.register("START")
def start(value):
    return "starting"


@command.register("STOP")
def stop(value):
    return "stopping"


assert command("start") == "starting"
assert command("START") == "starting"
assert command("Stop") == "stopping"
assert command("pause") == "unknown"

print("Problem 5 passed")

Problem 5 passed


### Explanation

Normalizing both registration and dispatch prevents duplicated cases such as `"START"`, `"start"`, and `"Start"`. The selected function receives the original value, not the normalized key.

## Problem 6 — Duplicate Case Protection

Modify the switch decorator so that registering the same case twice raises a `KeyError`.

Requirements:

- Registering a new case should work normally.
- Registering an already-used case should raise `KeyError`.
- The error message should include the duplicate case value.

This is useful when silent overwriting would hide bugs.

In [6]:
def strict_switcher(default_fn):
    registry = {"default": default_fn}

    def register(case):
        if case in registry:
            raise KeyError(f"Duplicate case registered: {case!r}")

        def inner(fn):
            registry[case] = fn
            return fn
        return inner

    def dispatch(case, *args, **kwargs):
        fn = registry.get(case, registry["default"])
        return fn(case, *args, **kwargs)

    dispatch.register = register
    dispatch.registry = registry
    return dispatch


@strict_switcher
def http_status(code):
    return "Unknown status"


@http_status.register(200)
def status_200(code):
    return "OK"


assert http_status(200) == "OK"
assert http_status(500) == "Unknown status"

try:
    @http_status.register(200)
    def duplicate_200(code):
        return "Duplicate OK"
except KeyError as ex:
    print(ex)
else:
    raise AssertionError("Duplicate registration should have failed")

print("Problem 6 passed")

'Duplicate case registered: 200'
Problem 6 passed


### Explanation

A plain dictionary assignment silently overwrites existing keys. For production-style dispatch tables, duplicate protection can catch accidental redefinition early.

## Problem 7 — Introspectable Switch

Build a switch decorator that exposes useful introspection methods.

Requirements:

The decorated dispatcher should provide:

- `.register(case)`
- `.cases()` returning registered non-default cases
- `.has_case(case)` returning `True` or `False`
- `.get_handler(case)` returning the handler function for a case, or the default handler if missing

Use this to build a small calculator dispatch system.

In [7]:
def introspectable_switcher(default_fn):
    registry = {"default": default_fn}

    def register(case):
        def inner(fn):
            registry[case] = fn
            return fn
        return inner

    def dispatch(case, *args, **kwargs):
        fn = registry.get(case, registry["default"])
        return fn(*args, **kwargs)

    def cases():
        return tuple(key for key in registry if key != "default")

    def has_case(case):
        return case in registry and case != "default"

    def get_handler(case):
        return registry.get(case, registry["default"])

    dispatch.register = register
    dispatch.cases = cases
    dispatch.has_case = has_case
    dispatch.get_handler = get_handler
    dispatch.registry = registry
    return dispatch


@introspectable_switcher
def calculate(x, y):
    raise ValueError("Unsupported operation")


@calculate.register("+")
def add(x, y):
    return x + y


@calculate.register("-")
def subtract(x, y):
    return x - y


@calculate.register("*")
def multiply(x, y):
    return x * y


assert calculate("+", 8, 2) == 10
assert calculate("-", 8, 2) == 6
assert calculate("*", 8, 2) == 16
assert set(calculate.cases()) == {"+", "-", "*"}
assert calculate.has_case("+") is True
assert calculate.has_case("/") is False
assert calculate.get_handler("+")(8, 2) == 10

print("Problem 7 passed")

Problem 7 passed


### Explanation

Adding introspection makes a dispatch system easier to test, debug, document, and expose in user-facing APIs. The switch is no longer just executable; it is inspectable.

## Problem 8 — Class-Based Switch Dispatcher

Implement a reusable class called `SwitchDispatcher`.

Requirements:

- Constructor accepts an optional default function.
- `.register(case, fn=None)` supports both direct registration and decorator-style registration.
- Calling the instance dispatches to the selected function.
- `.unregister(case)` removes a case.
- `.cases` property returns a tuple of registered cases.

The class should support both:

```python
dispatcher.register("a", some_function)
```

and:

```python
@dispatcher.register("b")
def handler(...):
    ...
```

In [8]:
class SwitchDispatcher:
    def __init__(self, default=None):
        self._registry = {}
        self._default = default or self._missing_case

    @staticmethod
    def _missing_case(*args, **kwargs):
        raise KeyError("No matching case and no default handler provided")

    def register(self, case, fn=None):
        if fn is not None:
            self._registry[case] = fn
            return fn

        def decorator(fn):
            self._registry[case] = fn
            return fn

        return decorator

    def unregister(self, case):
        del self._registry[case]

    @property
    def cases(self):
        return tuple(self._registry.keys())

    def __call__(self, case, *args, **kwargs):
        fn = self._registry.get(case, self._default)
        return fn(*args, **kwargs)


def default_formatter(value):
    return f"Unsupported format: {value!r}"


formatter = SwitchDispatcher(default=default_formatter)


@formatter.register("upper")
def upper(value):
    return value.upper()


def lower(value):
    return value.lower()


formatter.register("lower", lower)

assert formatter("upper", "Python") == "PYTHON"
assert formatter("lower", "Python") == "python"
assert formatter("title", "Python") == "Unsupported format: 'Python'"
assert set(formatter.cases) == {"upper", "lower"}

formatter.unregister("lower")
assert formatter.cases == ("upper",)

print("Problem 8 passed")

Problem 8 passed


### Explanation

A class-based dispatcher is more extensible than a closure when the switch needs additional features such as unregistering cases, exposing properties, or storing configuration.

## Problem 9 — Nested Dispatch for Command Subcommands

Create a two-level command dispatcher.

Requirements:

- First level dispatches on the main command: `"user"`, `"project"`.
- Second level dispatches on a subcommand.
- Supported commands:
  - `user create`
  - `user delete`
  - `project create`
  - `project archive`
- Unknown main commands should return `"Unknown command"`.
- Unknown subcommands should return `"Unknown subcommand"`.

Example:

```python
dispatch("user", "create", name="Ada")
```

In [9]:
def make_subcommand_router():
    def unknown_subcommand(**kwargs):
        return "Unknown subcommand"

    user_commands = {
        "create": lambda **kwargs: f"Created user {kwargs['name']}",
        "delete": lambda **kwargs: f"Deleted user {kwargs['name']}"
    }

    project_commands = {
        "create": lambda **kwargs: f"Created project {kwargs['name']}",
        "archive": lambda **kwargs: f"Archived project {kwargs['name']}"
    }

    routers = {
        "user": user_commands,
        "project": project_commands
    }

    def dispatch(command, subcommand, **kwargs):
        subcommands = routers.get(command)

        if subcommands is None:
            return "Unknown command"

        handler = subcommands.get(subcommand, unknown_subcommand)
        return handler(**kwargs)

    return dispatch


dispatch = make_subcommand_router()

assert dispatch("user", "create", name="Ada") == "Created user Ada"
assert dispatch("user", "delete", name="Ada") == "Deleted user Ada"
assert dispatch("project", "create", name="Compiler") == "Created project Compiler"
assert dispatch("project", "archive", name="Compiler") == "Archived project Compiler"
assert dispatch("team", "create", name="Core") == "Unknown command"
assert dispatch("user", "rename", name="Ada") == "Unknown subcommand"

print("Problem 9 passed")

Problem 9 passed


### Explanation

Nested dictionaries are useful when the dispatch decision has more than one dimension. In this case, the first dimension is the command and the second dimension is the subcommand.

## Problem 10 — Data-Driven Dispatch with Validation

You are given a list of event dictionaries. Each event has a `type` field.

Write a dispatcher that processes events of these types:

- `"login"`
- `"logout"`
- `"purchase"`

Requirements:

- Use dictionary-based dispatch.
- Validate required fields for each event type.
- Return a list of processed messages.
- Unknown event types should not crash the program.
- Malformed events should produce helpful error messages.

This problem combines switch-like dispatch with robust input handling.

In [10]:
def require_fields(event, *fields):
    missing = [field for field in fields if field not in event]
    if missing:
        raise ValueError(f"Missing required field(s): {', '.join(missing)}")


def handle_login(event):
    require_fields(event, "user")
    return f"User {event['user']} logged in"


def handle_logout(event):
    require_fields(event, "user")
    return f"User {event['user']} logged out"


def handle_purchase(event):
    require_fields(event, "user", "amount")
    return f"User {event['user']} purchased ${event['amount']:.2f}"


def handle_unknown(event):
    return f"Unknown event type: {event.get('type')!r}"


EVENT_HANDLERS = {
    "login": handle_login,
    "logout": handle_logout,
    "purchase": handle_purchase
}


def process_events(events):
    messages = []

    for index, event in enumerate(events):
        if not isinstance(event, dict):
            messages.append(f"Event {index}: event must be a dictionary")
            continue

        handler = EVENT_HANDLERS.get(event.get("type"), handle_unknown)

        try:
            messages.append(handler(event))
        except Exception as ex:
            messages.append(f"Event {index}: {ex}")

    return messages


events = [
    {"type": "login", "user": "Ada"},
    {"type": "purchase", "user": "Ada", "amount": 19.99},
    {"type": "logout", "user": "Ada"},
    {"type": "purchase", "user": "Linus"},
    {"type": "signup", "user": "Grace"},
    "not a dictionary"
]

result = process_events(events)

expected = [
    "User Ada logged in",
    "User Ada purchased $19.99",
    "User Ada logged out",
    "Event 3: Missing required field(s): amount",
    "Unknown event type: 'signup'",
    "Event 5: event must be a dictionary"
]

assert result == expected

for message in result:
    print(message)

print("Problem 10 passed")

User Ada logged in
User Ada purchased $19.99
User Ada logged out
Event 3: Missing required field(s): amount
Unknown event type: 'signup'
Event 5: event must be a dictionary
Problem 10 passed


### Explanation

In realistic applications, dispatch is only part of the problem. The code must also validate data, handle missing fields, and avoid crashing on unexpected input.

## Problem 11 — Switch Dispatcher with Method Binding

Create a class called `TicketMachine` that uses dictionary dispatch internally.

Requirements:

- The machine has a mutable `balance`.
- Supported actions:
  - `"insert"`: add money
  - `"refund"`: reset balance to zero and return refunded amount
  - `"buy"`: buy a ticket if balance is sufficient
- Use bound methods in the dispatch dictionary.
- Unknown actions should raise `ValueError`.

Ticket price is `2.50`.

In [11]:
class TicketMachine:
    PRICE = 2.50

    def __init__(self):
        self.balance = 0.0

    def insert(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive")
        self.balance += amount
        return self.balance

    def refund(self):
        amount = self.balance
        self.balance = 0.0
        return amount

    def buy(self):
        if self.balance < self.PRICE:
            return "Insufficient balance"

        self.balance -= self.PRICE
        return "Ticket purchased"

    def handle(self, action, *args):
        actions = {
            "insert": self.insert,
            "refund": self.refund,
            "buy": self.buy
        }

        try:
            handler = actions[action]
        except KeyError:
            raise ValueError(f"Unknown action: {action!r}")

        return handler(*args)


machine = TicketMachine()

assert machine.handle("insert", 1.00) == 1.00
assert machine.handle("buy") == "Insufficient balance"
assert machine.handle("insert", 2.00) == 3.00
assert machine.handle("buy") == "Ticket purchased"
assert machine.balance == 0.50
assert machine.handle("refund") == 0.50
assert machine.balance == 0.0

try:
    machine.handle("dance")
except ValueError as ex:
    print(ex)
else:
    raise AssertionError("Unknown action should raise ValueError")

print("Problem 11 passed")

Unknown action: 'dance'
Problem 11 passed


### Explanation

Dictionary values can be bound methods. A bound method already remembers its instance, so `self.insert` can be stored and called later without explicitly passing `self`.

## Problem 12 — Plugin-Style Dispatch System

Build a plugin registry for exporting data.

Requirements:

- Create an `ExporterRegistry` class.
- It should allow exporters to be registered by format name.
- It should reject duplicate formats unless `replace=True` is provided.
- It should support `export(format_name, data)`.
- Implement exporters for `"csv"`, `"json"`, and `"lines"`.

This is a more advanced version of switch-like dispatch because new behavior can be installed dynamically.

In [12]:
import json


class ExporterRegistry:
    def __init__(self):
        self._exporters = {}

    def register(self, format_name, fn=None, *, replace=False):
        normalized = format_name.lower()

        def add_exporter(exporter_fn):
            if normalized in self._exporters and not replace:
                raise KeyError(f"Exporter already registered: {normalized!r}")
            self._exporters[normalized] = exporter_fn
            return exporter_fn

        if fn is not None:
            return add_exporter(fn)

        return add_exporter

    def export(self, format_name, data):
        normalized = format_name.lower()

        try:
            exporter = self._exporters[normalized]
        except KeyError:
            raise ValueError(f"Unsupported export format: {format_name!r}")

        return exporter(data)

    @property
    def formats(self):
        return tuple(sorted(self._exporters))


exporters = ExporterRegistry()


@exporters.register("json")
def export_json(data):
    return json.dumps(data, sort_keys=True)


@exporters.register("lines")
def export_lines(data):
    return "\n".join(str(item) for item in data)


def export_csv(rows):
    if not rows:
        return ""

    headers = list(rows[0].keys())
    output = [",".join(headers)]

    for row in rows:
        output.append(",".join(str(row.get(header, "")) for header in headers))

    return "\n".join(output)


exporters.register("csv", export_csv)

rows = [
    {"name": "Ada", "score": 95},
    {"name": "Grace", "score": 98}
]

assert exporters.export("json", {"b": 2, "a": 1}) == '{"a": 1, "b": 2}'
assert exporters.export("lines", [1, 2, 3]) == "1\n2\n3"
assert exporters.export("csv", rows) == "name,score\nAda,95\nGrace,98"
assert exporters.formats == ("csv", "json", "lines")

try:
    exporters.register("csv", export_csv)
except KeyError as ex:
    print(ex)
else:
    raise AssertionError("Duplicate exporter should fail")

print("Problem 12 passed")

"Exporter already registered: 'csv'"
Problem 12 passed


### Explanation

A plugin registry is a production-grade version of a switch. Instead of editing a large conditional block, new behavior can be registered from elsewhere in the program.

## Best-Practice Summary

Use `if/elif/else` when:

- Branch logic is complex.
- Conditions are not simple equality checks.
- Readability would be worse with indirection.

Use dictionary dispatch when:

- Cases are simple equality checks.
- Each case maps cleanly to a value or function.
- You want fast, compact lookup behavior.

Use decorator or registry dispatch when:

- Cases should be registered dynamically.
- You want plugin-like extensibility.
- You want each case implementation to live near its function definition.

Avoid:

- Silent duplicate registrations unless overwriting is intentional.
- Huge anonymous lambdas that hurt readability.
- Hiding complex business rules behind overly clever dispatch tables.
- Returning side effects when returning values would be easier to test.